In [2]:
import tensorflow as tf
import cv2
import numpy as np
import albumentations as A
from sklearn.model_selection import train_test_split

print("TensorFlow version:", tf.__version__)
print("GPU tersedia:", tf.config.list_physical_devices('GPU'))

# Cek MobileNetV2 bisa diload
model = tf.keras.applications.MobileNetV2(weights='imagenet')
print("MobileNetV2 berhasil diload!")

TensorFlow version: 2.21.0
GPU tersedia: []
MobileNetV2 berhasil diload!


## 1. Load Dataset Menggunakan `image_dataset_from_directory` karena data (dan augmentasinya) sudah tersedia di folder masing-masing.

In [3]:
import os

dataset_dir = r"c:\Users\Tristan\Downloads\cleaned_dataset"
train_dir = os.path.join(dataset_dir, 'train')
val_dir = os.path.join(dataset_dir, 'val')
test_dir = os.path.join(dataset_dir, 'test')

BATCH_SIZE = 32
IMG_SIZE = (224, 224)

train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    shuffle=True,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE
)

val_dataset = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    shuffle=True,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE
)

test_dataset = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    shuffle=False,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE
)

class_names = train_dataset.class_names
print("Class names:", class_names)


Found 18000 files belonging to 6 classes.
Found 2040 files belonging to 6 classes.
Found 2040 files belonging to 6 classes.
Class names: ['freshapples', 'freshbanana', 'freshoranges', 'rottenapples', 'rottenbanana', 'rottenoranges']


## 2. Prefetching & Preprocessing\nMeningkatkan performa I/O dataset dan menerapkan preprocessing bawaan MobileNetV2.

In [4]:
AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)
val_dataset = val_dataset.prefetch(buffer_size=AUTOTUNE)
test_dataset = test_dataset.prefetch(buffer_size=AUTOTUNE)

preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input


## 3. Membangun Arsitektur Model

In [5]:
# Buat base model MobileNetV2
base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights='imagenet'
)

# Freeze base model (kita tidak melatih layer dasar terlebih dahulu)
base_model.trainable = False

# Layer Arsitektur Klasifikasi Kustom
inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = preprocess_input(inputs) # Preprocessing spesifik MobileNetV2
x = base_model(x, training=False) # Pastikan BatchNorm layer jalan di inference mode
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(6, activation='softmax')(x) # 6 kelas buah

model = tf.keras.Model(inputs, outputs)

model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 6)              │         7,686 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,265,670 (8.64 MB)

 Trainable params: 7,686 (30.02 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

## 4. Kompilasi & Pelatihan Model

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

# custom callback
class TargetCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs={}):
        if(logs.get('accuracy') >= 0.95 and logs.get('val_accuracy') >= 0.98):
            print("menghentikan proses training")
            self.model.stop_training = True

my_callback = TargetCallback()

# Setup Callbacks
callbacks = [
    my_callback,
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint('best_mobilenet_model.keras', monitor='val_accuracy', save_best_only=True)
]

# Jalankan training
initial_epochs = 30

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=initial_epochs,
    callbacks=callbacks
)


Epoch 1/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 113s 196ms/step - accuracy: 0.6912 - loss: 0.9002 - val_accuracy: 0.9402 - val_loss: 0.3103
Epoch 2/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 104s 185ms/step - accuracy: 0.8934 - loss: 0.3507 - val_accuracy: 0.9627 - val_loss: 0.1762
Epoch 3/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 111s 197ms/step - accuracy: 0.9189 - loss: 0.2583 - val_accuracy: 0.9721 - val_loss: 0.1315
Epoch 4/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 119s 211ms/step - accuracy: 0.9301 - loss: 0.2153 - val_accuracy: 0.9755 - val_loss: 0.1069
Epoch 5/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 119s 211ms/step - accuracy: 0.9408 - loss: 0.1877 - val_accuracy: 0.9779 - val_loss: 0.0924
Epoch 6/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 120s 213ms/step - accuracy: 0.9451 - loss: 0.1683 - val_accuracy: 0.9819 - val_loss: 0.0810
Epoch 7/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 120s 213ms/step - accuracy: 0.9496 - loss: 0.1536 - val_accuracy: 0.9828 - val_loss: 0.0742
Epoch 8/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 0s 200ms/step - accuracy: 0.9529 - l

## 5. Evaluasi & Ekspor ke .keras

In [7]:
# Evaluasi di data testing

loss, accuracy = model.evaluate(test_dataset)
print(f'Test accuracy: {accuracy:.4f}')

# Simpan model untuk dikonversi ke TFJS
model_save_path = 'mobilenet_fruit_model.keras'
model.save(model_save_path)
print(f"Model berhasil disimpan di: {model_save_path}")


64/64 ━━━━━━━━━━━━━━━━━━━━ 12s 185ms/step - accuracy: 0.9877 - loss: 0.0596
Test accuracy: 0.9877
Model berhasil disimpan di: mobilenet_fruit_model.keras
